# MarsaTrack AI - Entrainement YOLO11 container_code

Objectif : entrainer un modele YOLO pour localiser la zone du matricule ISO 6346 sur une image de conteneur.

YOLO detecte la zone. L'OCR lira ensuite les caracteres dans cette zone. La validation ISO 6346 verifiera enfin le code lu.

## 1. Verification de l'environnement Kaggle

In [ ]:
import os
from pathlib import Path

print('Kaggle input:', os.listdir('/kaggle/input') if Path('/kaggle/input').exists() else 'hors Kaggle')
print('Kaggle working:', Path('/kaggle/working').exists())

## 2. Installation d'Ultralytics

Si Kaggle contient deja Ultralytics, cette cellule peut etre rapide. YOLO11 est fourni via le package `ultralytics`.

In [ ]:
!pip install -q ultralytics

## 3. Imports

In [ ]:
import random
import shutil
from pathlib import Path

import torch
from IPython.display import Image, display
from ultralytics import YOLO

random.seed(42)

## 4. Definition des chemins

Adapter `DATASET_ROOT` selon le nom du dataset importe dans Kaggle. Le fichier YAML ne doit pas utiliser de chemin Windows.

In [ ]:
DATASET_ROOT = Path('/kaggle/working/container-code-dataset')
DATA_YAML = DATASET_ROOT / 'container_code.yaml'
RUNS_DIR = Path('/kaggle/working/runs')
EXPORT_DIR = Path('/kaggle/working/marsatrack-export')

print('Dataset root:', DATASET_ROOT)
print('YAML:', DATA_YAML)

## 5. Verification GPU CUDA

In [ ]:
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Activer un GPU Kaggle si possible pour accelerer l entrainement.')

## 6. Verification du dataset

Le dataset doit contenir `images/train`, `images/val`, `images/test`, et les labels correspondants.

In [ ]:
for split in ['train', 'val', 'test']:
    image_dir = DATASET_ROOT / 'images' / split
    label_dir = DATASET_ROOT / 'labels' / split
    images = list(image_dir.glob('*')) if image_dir.exists() else []
    labels = list(label_dir.glob('*.txt')) if label_dir.exists() else []
    print(f'{split}: {len(images)} images, {len(labels)} labels')

assert DATA_YAML.exists(), f'YAML introuvable: {DATA_YAML}'

## 7. Affichage d'exemples annotees

Apres export Roboflow, verifier visuellement quelques images annotees avant entrainement.

In [ ]:
from PIL import Image as PILImage, ImageDraw, ImageFont

CLASS_NAME = 'container_code'
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}


def read_yolo_annotations(label_path):
    if not label_path.exists():
        print(f'Avertissement : label manquant pour {label_path.name}')
        return None

    content = label_path.read_text(encoding='utf-8').strip()
    if not content:
        return []

    annotations = []
    for line_number, raw_line in enumerate(content.splitlines(), start=1):
        parts = raw_line.strip().split()
        if len(parts) != 5:
            print(f'Avertissement : ligne YOLO invalide dans {label_path.name}:{line_number}')
            continue
        class_id, x_center, y_center, width, height = parts
        if class_id != '0':
            print(f'Avertissement : classe inattendue {class_id} dans {label_path.name}:{line_number}')
            continue
        annotations.append(tuple(float(value) for value in (x_center, y_center, width, height)))
    return annotations


def draw_yolo_boxes(image_path, label_path):
    with PILImage.open(image_path).convert('RGB') as image:
        draw = ImageDraw.Draw(image)
        image_width, image_height = image.size
        annotations = read_yolo_annotations(label_path)

        if annotations is None:
            draw.text((16, 16), 'LABEL MANQUANT', fill=(220, 38, 38), font=ImageFont.load_default())
            return image

        if not annotations:
            draw.text((16, 16), 'Image negative : aucune zone container_code annotee', fill=(90, 113, 143), font=ImageFont.load_default())
            return image

        for x_center, y_center, width, height in annotations:
            box_width = width * image_width
            box_height = height * image_height
            x1 = x_center * image_width - box_width / 2
            y1 = y_center * image_height - box_height / 2
            x2 = x_center * image_width + box_width / 2
            y2 = y_center * image_height + box_height / 2

            draw.rectangle((x1, y1, x2, y2), outline=(0, 153, 204), width=4)
            label = CLASS_NAME
            text_bbox = draw.textbbox((x1, y1), label, font=ImageFont.load_default())
            text_width = text_bbox[2] - text_bbox[0]
            text_height = text_bbox[3] - text_bbox[1]
            label_y1 = max(0, y1 - text_height - 8)
            draw.rectangle((x1, label_y1, x1 + text_width + 10, label_y1 + text_height + 8), fill=(0, 56, 130))
            draw.text((x1 + 5, label_y1 + 4), label, fill=(255, 255, 255), font=ImageFont.load_default())

        return image


train_image_dir = DATASET_ROOT / 'images' / 'train'
train_label_dir = DATASET_ROOT / 'labels' / 'train'
sample_images = sorted(
    image_path for image_path in train_image_dir.glob('*')
    if image_path.suffix.lower() in IMAGE_EXTENSIONS
)[:6]

if len(sample_images) < 6:
    print(f'Avertissement : seulement {len(sample_images)} exemple(s) disponible(s) dans le split train.')

for image_path in sample_images:
    print(f'Exemple annote : {image_path.name}')
    annotated = draw_yolo_boxes(image_path, train_label_dir / f'{image_path.stem}.txt')
    display(annotated)


## 8. Chargement de YOLO11

Modele initial recommande : `yolo11n.pt`, leger et rapide pour un premier prototype.

In [ ]:
model = YOLO('yolo11n.pt')

## 9. Entrainement

Parametres de depart : `epochs=50`, `imgsz=640`, `batch=-1`, `patience=15`.

Si Kaggle manque de memoire, reduire `batch`, `imgsz` ou `workers`.

In [ ]:
results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=-1,
    patience=15,
    device=0,
    workers=2,
    project=str(RUNS_DIR),
    name='container_code_yolo11n',
    seed=42,
    plots=True,
    save=True,
)

In [ ]:
# Ultralytics peut creer un dossier incremente, par exemple container_code_yolo11n2.
# On recupere donc le chemin reel depuis l'objet results, avec fallback vers le chemin attendu.
run_dir = Path(getattr(results, 'save_dir', RUNS_DIR / 'container_code_yolo11n'))
if not run_dir.exists():
    fallback_run_dir = RUNS_DIR / 'container_code_yolo11n'
    print(f'Avertissement : save_dir introuvable ({run_dir}). Fallback vers {fallback_run_dir}.')
    run_dir = fallback_run_dir

print('Dossier reel du run Ultralytics:', run_dir)


## 10. Evaluation validation

In [ ]:
val_metrics = model.val(data=str(DATA_YAML), split='val')
print(val_metrics)

## 11. Evaluation test

Le split test doit rester independant pour mesurer la generalisation.

In [ ]:
test_metrics = model.val(data=str(DATA_YAML), split='test')
print(test_metrics)

## 12. Affichage des metriques

A relever : Precision, Recall, mAP50, mAP50-95, matrice de confusion, courbes Precision-Recall.

Ne pas conclure que le modele est bon uniquement parce que la loss diminue.

In [ ]:
for plot_name in ['results.png', 'confusion_matrix.png', 'PR_curve.png']:
    plot_path = run_dir / plot_name
    if plot_path.exists():
        display(Image(filename=str(plot_path), width=900))
    else:
        print('Plot absent:', plot_path)


## 13. Predictions sur quelques images

Inspecter visuellement les vrais positifs, faux positifs et faux negatifs.

In [ ]:
test_images = list((DATASET_ROOT / 'images' / 'test').glob('*'))[:8]
predictions = model.predict(source=[str(path) for path in test_images], imgsz=640, conf=0.25, save=True)
print('Predictions generees:', len(predictions))

## 14. Export du meilleur modele

Le fichier principal attendu est `best.pt`. L'export ONNX est optionnel.

In [ ]:
BEST_MODEL_PATH = run_dir / 'weights' / 'best.pt'
assert BEST_MODEL_PATH.exists(), f'best.pt introuvable: {BEST_MODEL_PATH}'

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
target = EXPORT_DIR / 'container_code_best.pt'
shutil.copy2(BEST_MODEL_PATH, target)
print('Modele copie vers:', target)

## 15. Export ONNX optionnel et sauvegarde

ONNX peut etre utile plus tard, mais `best.pt` reste le livrable principal de cette etape.

In [ ]:
# Optionnel : decommenter si un export ONNX est souhaite.
# best_model = YOLO(str(BEST_MODEL_PATH))
# best_model.export(format='onnx')

print('Telecharger ensuite:', target)